# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [15]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "indonesia_palm_oil_v1_2_5.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "dataset_bersih.parquet"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}"
    )

print(f"Lokasi dataset: {DATA_PATH}")
print(
    f"Ukuran file: "
    f"{DATA_PATH.stat().st_size / (1024**2):,.2f} MB"
)

df = pl.read_csv(DATA_PATH)

print(f"Jumlah baris awal: {df.height:,}")
print(f"Jumlah kolom awal: {df.width:,}")

display(df.head())


Lokasi dataset: /home/jovyan/work/data/raw/indonesia_palm_oil_v1_2_5.csv
Ukuran file: 721.83 MB
Jumlah baris awal: 1,694,907
Jumlah kolom awal: 34


year,country_of_production,product_type,forest_500_palm_oil,zero_deforestation_indonesia_palm_oil,province_of_production,kabupaten_of_production,mill,mill_trase_id,mill_group,refinery,refinery_trase_id,refinery_group,exporter,exporter_group,port_of_export,importer,importer_group,country_of_first_import,economic_bloc,province_of_production_trase_id,kabupaten_of_production_trase_id,country_of_production_trase_id,country_of_first_import_trase_id,palm_oil_deforestation_10_year_total_exposure,palm_oil_deforestation_annual_exposure,total_emission_exposure,net_emission_luc_exposure,emission_subsidence_exposure,gross_emission_luc_exposure,emissions_fire_on_peat_exposure,volume,oil_palm_ha,fob
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,str
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ACCESS BALI""","""ACCESS BALI""","""TANJUNG PRIOK""","""JOV-HENTON LTD""","""JOV-HENTON LTD""","""NIGERIA""","""NIGERIA""","""ID-XX""","""ID-XXXX""","""ID""","""NG""",null,null,null,null,null,null,null,22.8,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""ACALPO""","""ACALPO""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,3000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""BUNGE""","""BUNGE""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,1000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""CARGILL""","""CARGILL""","""INDIA""","""INDIA""","""ID-XX""","""ID-XXXX""","""ID""","""IN""",null,null,null,null,null,null,null,1000.0,null,null
2013,"""INDONESIA""","""PALM OIL""","""NOT ASSESSED""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-MILL-X""","""UNKNOWN""","""UNKNOWN""","""ID-PALM-REFINERY-X""","""UNKNOWN""","""ADEI PLANTATION & INDUSTRY""","""KUALA LUMPUR KEPONG (KLK)""","""DUMAI""","""CARGILL""","""CARGILL""","""TANZANIA""","""TANZANIA""","""ID-XX""","""ID-XXXX""","""ID""","""TZ""",null,null,null,null,null,null,null,500.0,null,null


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

Tahap pertama adalah melakukan *data profiling* menggunakan fungsi `SUMMARIZE` dari DuckDB. Fungsi ini menghasilkan ringkasan karakteristik setiap kolom, termasuk tipe data, jumlah nilai yang berbeda, jumlah nilai kosong, serta statistik deskriptif untuk kolom yang relevan.

Profiling digunakan untuk memperoleh gambaran awal mengenai struktur dan kualitas dataset sebelum dilakukan pemeriksaan missing values secara lebih mendalam. Hasilnya juga membantu mengidentifikasi kolom yang memerlukan perhatian pada tahap pembersihan data.

Pada tahap ini, data hanya diperiksa dan diringkas. DataFrame asli tidak diubah.

In [16]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""year""","""BIGINT""","""2013""","""2022""",11,"""2020.0519167128343""","""1.354049969056122""","""2019""","""2020""","""2021""",1694907,0.00
"""country_of_production""","""VARCHAR""","""INDONESIA""","""INDONESIA""",1,null,null,null,null,null,1694907,0.00
"""product_type""","""VARCHAR""","""PALM OIL""","""REFINED PALM OIL""",2,null,null,null,null,null,1694907,0.00
"""forest_500_palm_oil""","""VARCHAR""","""0""","""NOT ASSESSED""",7,null,null,null,null,null,1694907,0.00
"""zero_deforestation_indonesia_p…","""VARCHAR""","""COMPANY COMMITMENT""","""UNKNOWN""",4,null,null,null,null,null,1694907,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""gross_emission_luc_exposure""","""VARCHAR""","""0""","""999.9926319638474""",142065,null,null,null,null,null,1694907,60.49
"""emissions_fire_on_peat_exposur…","""VARCHAR""","""0""","""997.5440867831901""",51306,null,null,null,null,null,1694907,60.49
"""volume""","""DOUBLE""","""5.041356112552372e-11""","""11605224.752096057""",1619108,"""226.34659706787994""","""15280.810851364538""","""0.1534665478718176""","""2.129528406099078""","""21.225549395764002""",1694907,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

### 2.1. Identifikasi Missing Values

Setelah memperoleh gambaran umum dataset, langkah berikutnya adalah menghitung jumlah dan persentase nilai kosong pada setiap kolom. Pemeriksaan ini dilakukan untuk mengetahui kolom mana yang memiliki missing values dan seberapa besar proporsinya terhadap keseluruhan data.

Persentase missing values dihitung dengan membandingkan jumlah nilai kosong pada suatu kolom dengan jumlah seluruh baris, kemudian dikalikan 100%. Kolom yang tidak memiliki nilai kosong tidak ditampilkan agar perhatian dapat difokuskan pada kolom yang memerlukan pemeriksaan lebih lanjut.

Hasil pemeriksaan ini menjadi dasar untuk menentukan kolom yang perlu dianalisis pada tahap berikutnya.

In [17]:
# Hitung jumlah nilai null pada setiap kolom
missing_summary = pl.DataFrame({
    "column_name": df.columns,
    "null_count": [
        df[col].null_count()
        for col in df.columns
    ],
    "null_percentage": [
        round(
            df[col].null_count() / df.height * 100,
            2
        )
        for col in df.columns
    ],
})

# Tampilkan hanya kolom yang memiliki nilai null
missing_summary = (
    missing_summary
    .filter(pl.col("null_count") > 0)
    .sort("null_percentage", descending=True)
)

print("Jumlah baris sebelum penanganan missing values:", f"{df.height:,}")
print("Jumlah kolom yang memiliki nilai null:", missing_summary.height)

display(missing_summary)


Jumlah baris sebelum penanganan missing values: 1,694,907
Jumlah kolom yang memiliki nilai null: 9


column_name,null_count,null_percentage
str,i64,f64
"""total_emission_exposure""",1025290,60.49
"""net_emission_luc_exposure""",1025290,60.49
"""emission_subsidence_exposure""",1025290,60.49
"""gross_emission_luc_exposure""",1025290,60.49
"""emissions_fire_on_peat_exposur…",1025290,60.49
"""fob""",10957,0.65
"""palm_oil_deforestation_10_year…",10233,0.6
"""palm_oil_deforestation_annual_…",10233,0.6
"""oil_palm_ha""",10233,0.6


### 2.2. Pemeriksaan Tipe Data dan Contoh Nilai

Kolom yang teridentifikasi memiliki missing values selanjutnya diperiksa tipe datanya dan beberapa contoh nilai yang tersedia. Pemeriksaan ini diperlukan untuk memahami apakah nilai dalam kolom sudah direpresentasikan sesuai karakteristik datanya.

Pemeriksaan menjadi penting ketika kolom yang secara konseptual berisi angka justru terbaca sebagai teks (`String`). Kondisi tersebut dapat memengaruhi proses perhitungan dan analisis numerik apabila tidak dikenali sejak awal.

Pada tahap ini, tipe data dan contoh nilai diperiksa tanpa melakukan konversi atau perubahan terhadap DataFrame asli.

In [18]:
# PEMERIKSAAN KOLOM YANG MEMILIKI MISSING VALUES
missing_columns = missing_summary["column_name"].to_list()

# Tampilkan tipe data untuk setiap kolom yang bermasalah
type_summary = pl.DataFrame({
    "column_name": missing_columns,
    "data_type": [
        str(df.schema[col])
        for col in missing_columns
    ],
})

print("Tipe data kolom yang memiliki missing values:")
display(type_summary)

# Tampilkan beberapa contoh nilai dari setiap kolom
for col in missing_columns:
    print(f"\nKolom: {col}")
    print(f"Tipe data: {df.schema[col]}")
    print(f"Jumlah null: {df[col].null_count():,}")

    display(
        df.select(col)
        .filter(pl.col(col).is_not_null())
        .head(5)
    )


Tipe data kolom yang memiliki missing values:


column_name,data_type
str,str
"""total_emission_exposure""","""String"""
"""net_emission_luc_exposure""","""String"""
"""emission_subsidence_exposure""","""String"""
"""gross_emission_luc_exposure""","""String"""
"""emissions_fire_on_peat_exposur…","""String"""
"""fob""","""String"""
"""palm_oil_deforestation_10_year…","""String"""
"""palm_oil_deforestation_annual_…","""String"""
"""oil_palm_ha""","""String"""



Kolom: total_emission_exposure
Tipe data: String
Jumlah null: 1,025,290


total_emission_exposure
str
"""0"""
"""0"""
"""0"""
"""0"""
"""0"""



Kolom: net_emission_luc_exposure
Tipe data: String
Jumlah null: 1,025,290


net_emission_luc_exposure
str
"""0"""
"""0"""
"""0"""
"""0"""
"""0"""


... (10 outputs left out)


Kolom: palm_oil_deforestation_annual_exposure
Tipe data: String
Jumlah null: 10,233


palm_oil_deforestation_annual_exposure
str
"""0"""
"""0"""
"""0"""
"""0"""
"""0"""



Kolom: oil_palm_ha
Tipe data: String
Jumlah null: 10,233


oil_palm_ha
str
"""187.57247469148024"""
"""31.28815257572648"""
"""25.03052206058118"""
"""3.1536589396105703"""
"""242.3898299281281"""


### 2.3. Pemeriksaan Teks Kosong dan Validitas Nilai Numerik

Nilai kosong dapat direpresentasikan sebagai `null` maupun teks kosong. Selain itu, kolom bertipe teks yang seharusnya berisi angka mungkin mengandung nilai yang tidak dapat dikonversi menjadi numerik.

Untuk memeriksa kemungkinan tersebut, setiap kolom yang memiliki missing values diperiksa terhadap teks kosong dan nilai yang gagal dikonversi ke format numerik. Konversi pada tahap diagnostik ini hanya digunakan untuk menguji validitas nilai, bukan untuk mengubah tipe data pada DataFrame asli.

Hasil pemeriksaan membantu membedakan missing values yang benar-benar tersimpan sebagai `null` dari kemungkinan masalah format atau nilai yang tidak valid.

In [19]:
# DIAGNOSTIK FORMAT NILAI
diagnostic_rows = []

for col in missing_columns:
    # Buat DataFrame sementara dengan nama kolom yang konsisten
    temp = df.select(
        pl.col(col)
        .cast(pl.String)
        .str.strip_chars()
        .alias("value")
    )

    raw = pl.col("value")
    numeric = raw.cast(pl.Float64, strict=False)

    # Jumlah string kosong atau hanya spasi
    blank_count = temp.select(
        (raw == "").fill_null(False).sum()
    ).item()

    # Jumlah nilai non-null yang gagal dikonversi menjadi angka
    invalid_condition = (
        raw.is_not_null()
        & numeric.is_null()
        & (raw != "").fill_null(False)
    )

    invalid_count = temp.select(
        invalid_condition.sum()
    ).item()

    # Contoh nilai yang gagal dikonversi
    invalid_samples = (
        temp.filter(invalid_condition)
        .get_column("value")
        .unique()
        .head(5)
        .to_list()
    )

    diagnostic_rows.append({
        "column_name": col,
        "blank_text_count": blank_count,
        "invalid_numeric_count": invalid_count,
        "invalid_examples": str(invalid_samples),
    })

diagnostic_summary = pl.DataFrame(diagnostic_rows)

print("Pemeriksaan format nilai sebelum konversi:")
display(diagnostic_summary)

print(f"Jumlah baris tetap: {df.height:,}")
print("DataFrame asli belum diubah.")


Pemeriksaan format nilai sebelum konversi:


column_name,blank_text_count,invalid_numeric_count,invalid_examples
str,i64,i64,str
"""total_emission_exposure""",0,0,"""[]"""
"""net_emission_luc_exposure""",0,0,"""[]"""
"""emission_subsidence_exposure""",0,0,"""[]"""
"""gross_emission_luc_exposure""",0,0,"""[]"""
"""emissions_fire_on_peat_exposur…",0,0,"""[]"""
"""fob""",0,0,"""[]"""
"""palm_oil_deforestation_10_year…",0,0,"""[]"""
"""palm_oil_deforestation_annual_…",0,0,"""[]"""
"""oil_palm_ha""",0,0,"""[]"""


Jumlah baris tetap: 1,694,907
DataFrame asli belum diubah.


### 2.4. Analisis Pola Missing Values

Jumlah dan persentase missing values belum cukup untuk menjelaskan karakteristik nilai kosong dalam dataset. Oleh karena itu, pemeriksaan dilanjutkan dengan menganalisis distribusi missing values berdasarkan tahun (`year`) dan jenis produk (`product_type`).

Analisis dilakukan dengan menghitung jumlah baris dan jumlah nilai kosong pada lima kolom terkait emisi untuk setiap kelompok yang tersedia dalam dataset. Tujuannya adalah mengetahui apakah nilai kosong tersebar di berbagai kelompok atau terkonsentrasi pada periode dan jenis produk tertentu.

Hasil analisis digunakan untuk memahami pola ketersediaan data sebelum menentukan strategi penanganan missing values. Nilai kosong tidak langsung dianggap bernilai nol karena makna dan penyebab ketidaktersediaannya perlu dipertimbangkan terlebih dahulu.

Proses ini hanya menghasilkan ringkasan diagnostik dan tidak mengubah DataFrame asli.

In [20]:
group_columns = [
    col for col in [
        "year",
        "product_type",
        "flow_type",
        "country_of_destination",
    ]
    if col in df.columns
]

emission_columns = [
    col for col in [
        "total_emission_exposure",
        "net_emission_luc_exposure",
        "emission_subsidence_exposure",
        "gross_emission_luc_exposure",
        "emissions_fire_on_peat_exposure",
    ]
    if col in df.columns
]

print("Jumlah baris sebelum analisis:", f"{df.height:,}")
print("Kolom kategori yang tersedia:", group_columns)
print("Kolom emisi yang diperiksa:", emission_columns)

for group_col in group_columns:
    print(f"\nPola missing values berdasarkan: {group_col}")

    pattern_summary = (
        df.group_by(group_col)
        .agg([
            pl.len().alias("jumlah_baris"),
            *[
                pl.col(col).is_null().sum().alias(f"{col}_null")
                for col in emission_columns
            ],
        ])
        .sort(group_col)
    )

    display(pattern_summary)

print("\nDataFrame asli belum diubah.")
print("Jumlah baris setelah analisis:", f"{df.height:,}")


Jumlah baris sebelum analisis: 1,694,907
Kolom kategori yang tersedia: ['year', 'product_type']
Kolom emisi yang diperiksa: ['total_emission_exposure', 'net_emission_luc_exposure', 'emission_subsidence_exposure', 'gross_emission_luc_exposure', 'emissions_fire_on_peat_exposure']

Pola missing values berdasarkan: year


year,jumlah_baris,total_emission_exposure_null,net_emission_luc_exposure_null,emission_subsidence_exposure_null,gross_emission_luc_exposure_null,emissions_fire_on_peat_exposure_null
i64,u32,u32,u32,u32,u32,u32
2013,2095,2095,2095,2095,2095,2095
2014,2056,2056,2056,2056,2056,2056
2015,2038,2038,2038,2038,2038,2038
2016,2213,2213,2213,2213,2213,2213
2017,1831,1831,1831,1831,1831,1831
2018,247986,247986,247986,247986,247986,247986
2019,318343,318343,318343,318343,318343,318343
2020,448728,448728,448728,448728,448728,448728
2021,385389,0,0,0,0,0



Pola missing values berdasarkan: product_type


product_type,jumlah_baris,total_emission_exposure_null,net_emission_luc_exposure_null,emission_subsidence_exposure_null,gross_emission_luc_exposure_null,emissions_fire_on_peat_exposure_null
str,u32,u32,u32,u32,u32,u32
"""PALM OIL""",168053,129915,129915,129915,129915,129915
"""REFINED PALM OIL""",1526854,895375,895375,895375,895375,895375



DataFrame asli belum diubah.
Jumlah baris setelah analisis: 1,694,907


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [21]:
kolom_missing = [
    "total_emission_exposure",
    "net_emission_luc_exposure",
    "emission_subsidence_exposure",
    "gross_emission_luc_exposure",
    "emissions_fire_on_peat_exposure",
    "fob",
    "palm_oil_deforestation_10_year_total_exposure",
    "palm_oil_deforestation_annual_exposure",
    "oil_palm_ha",
]

# Simpan kondisi awal untuk perbandingan.
jumlah_baris_sebelum = df.height
null_sebelum = df.null_count().sum_horizontal().sum()

# Penyesuaian tipe data; nilai null tetap dipertahankan.
df_missing_handled = df.with_columns([
    pl.col(col).cast(pl.Float64, strict=False).alias(col)
    for col in kolom_missing
])

# Tampilkan perbandingan jumlah missing values sesuai template.
null_sesudah = (
    df_missing_handled.null_count()
    .sum_horizontal()
    .sum()
)

print(f"Null sebelum: {null_sebelum:,}")
print(f"Null sesudah: {null_sesudah:,}")

# Periksa perubahan tipe data pada kolom yang diproses.
display(
    pl.DataFrame({
        "kolom": kolom_missing,
        "tipe_data_sebelum": [str(df[col].dtype) for col in kolom_missing],
        "tipe_data_sesudah": [
            str(df_missing_handled[col].dtype) for col in kolom_missing
        ],
        "missing_sebelum": [df[col].null_count() for col in kolom_missing],
        "missing_sesudah": [
            df_missing_handled[col].null_count() for col in kolom_missing
        ],
    })
)

# Validasi hasil.
assert df_missing_handled.height == jumlah_baris_sebelum, (
    "Jumlah baris berubah setelah penanganan."
)

kolom_null_baru = [
    col for col in kolom_missing
    if df_missing_handled[col].null_count() > df[col].null_count()
]

assert not kolom_null_baru, (
    f"Konversi menghasilkan null baru pada: {kolom_null_baru}"
)

print(f"Jumlah baris sebelum: {jumlah_baris_sebelum:,}")
print(f"Jumlah baris sesudah: {df_missing_handled.height:,}")
print("Validasi berhasil: tidak ada null baru dan jumlah baris tetap.")


Null sebelum: 5,168,106
Null sesudah: 5,168,106


kolom,tipe_data_sebelum,tipe_data_sesudah,missing_sebelum,missing_sesudah
str,str,str,i64,i64
"""total_emission_exposure""","""String""","""Float64""",1025290,1025290
"""net_emission_luc_exposure""","""String""","""Float64""",1025290,1025290
"""emission_subsidence_exposure""","""String""","""Float64""",1025290,1025290
"""gross_emission_luc_exposure""","""String""","""Float64""",1025290,1025290
"""emissions_fire_on_peat_exposur…","""String""","""Float64""",1025290,1025290
"""fob""","""String""","""Float64""",10957,10957
"""palm_oil_deforestation_10_year…","""String""","""Float64""",10233,10233
"""palm_oil_deforestation_annual_…","""String""","""Float64""",10233,10233
"""oil_palm_ha""","""String""","""Float64""",10233,10233


Jumlah baris sebelum: 1,694,907
Jumlah baris sesudah: 1,694,907
Validasi berhasil: tidak ada null baru dan jumlah baris tetap.


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [24]:
# TODO: ganti 'kolom_numerik' dengan kolom numerik utama pada dataset Anda
KOLOM_NUMERIK = 'volume'

# Pastikan kolom tersedia dan bertipe numerik
if KOLOM_NUMERIK not in df_missing_handled.columns:
    raise ValueError(f"Kolom '{KOLOM_NUMERIK}' tidak ditemukan.")

if not df_missing_handled.schema[KOLOM_NUMERIK].is_numeric():
    raise TypeError(
        f"Kolom '{KOLOM_NUMERIK}' bukan bertipe numerik: "
        f"{df_missing_handled.schema[KOLOM_NUMERIK]}"
    )

# BEFORE: kondisi data sebelum penanganan
jumlah_baris_sebelum = df_missing_handled.height
print(f"Jumlah baris sebelum: {jumlah_baris_sebelum:,}")

# Menghitung batas outlier dengan metode IQR
q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

# Menghitung jumlah kandidat outlier sebelum penanganan
jumlah_outlier = df_missing_handled.filter(
    (pl.col(KOLOM_NUMERIK) < batas_bawah) |
    (pl.col(KOLOM_NUMERIK) > batas_atas)
).height

print(f"Jumlah kandidat outlier: {jumlah_outlier:,}")

# Windsorization: clip nilai ke batas yang valid, bukan menghapus barisnya
df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(
        lower_bound=batas_bawah,
        upper_bound=batas_atas
    )
)

# AFTER: memeriksa hasil penanganan
jumlah_baris_sesudah = df_outlier_handled.height

print(f"Jumlah baris sesudah: {jumlah_baris_sesudah:,}")
print(
    "Jumlah nilai yang berubah:",
    (
        (
            df_missing_handled[KOLOM_NUMERIK]
            != df_outlier_handled[KOLOM_NUMERIK]
        )
        .fill_null(False)
        .sum()
    )
)

# Validasi: winsorization tidak boleh menghapus baris
assert jumlah_baris_sesudah == jumlah_baris_sebelum

print("Validasi berhasil: jumlah baris tetap.")


Jumlah baris sebelum: 1,694,907
Q1=0.1521182186311248, Q3=21.21685539046564, IQR=21.064737171834516
Batas bawah: -31.44498753912065, Batas atas: 52.813961148217416
Jumlah kandidat outlier: 281,597
Jumlah baris sesudah: 1,694,907
Jumlah nilai yang berubah: 281597
Validasi berhasil: jumlah baris tetap.


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

Deduplikasi bertujuan mengidentifikasi dan menangani catatan yang tercatat lebih dari satu kali. Proses ini harus menggunakan kunci bisnis yang sesuai dengan karakteristik dataset, bukan secara otomatis menghapus baris yang memiliki seluruh atribut identik.

Dataset rantai pasok kelapa sawit yang digunakan tidak memiliki kolom ID baris yang telah terkonfirmasi sebagai kunci unik. Oleh karena itu, kandidat kombinasi atribut diperiksa terlebih dahulu sebelum keputusan penghapusan dilakukan.

In [29]:
# Kandidat 1: dimensi rantai pasok utama
KANDIDAT_KUNCI_1 = [
    "year",
    "product_type",
    "mill_trase_id",
    "refinery_trase_id",
    "exporter",
    "importer",
    "port_of_export",
    "country_of_first_import",
]

# Kandidat 2: seluruh atribut dimensi yang tersedia
# Tidak memasukkan kolom ukuran seperti volume, fob, dan emisi.
KANDIDAT_KUNCI_2 = [
    col for col in df_outlier_handled.columns
    if col not in [
        "volume",
        "oil_palm_ha",
        "fob",
        "palm_oil_deforestation_10_year_total_exposure",
        "palm_oil_deforestation_annual_exposure",
        "total_emission_exposure",
        "net_emission_luc_exposure",
        "emission_subsidence_exposure",
        "gross_emission_luc_exposure",
        "emissions_fire_on_peat_exposure",
    ]
]

# Memeriksa kandidat kunci tanpa mengubah data
hasil_kandidat = []

for nama, kolom_kunci in [
    ("Kandidat 1", KANDIDAT_KUNCI_1),
    ("Kandidat 2", KANDIDAT_KUNCI_2),
]:
    jumlah_kombinasi_unik = (
        df_outlier_handled
        .select(kolom_kunci)
        .unique()
        .height
    )

    hasil_kandidat.append({
        "kandidat": nama,
        "jumlah_kolom_kunci": len(kolom_kunci),
        "jumlah_kombinasi_unik": jumlah_kombinasi_unik,
        "jumlah_baris": df_outlier_handled.height,
        "selisih_baris_dan_kombinasi": (
            df_outlier_handled.height - jumlah_kombinasi_unik
        ),
    })

display(pl.DataFrame(hasil_kandidat))

print("Pemeriksaan selesai. Data belum dihapus atau diubah.")

kandidat,jumlah_kolom_kunci,jumlah_kombinasi_unik,jumlah_baris,selisih_baris_dan_kombinasi
str,i64,i64,i64,i64
"""Kandidat 1""",8,617082,1694907,1077825
"""Kandidat 2""",24,1694907,1694907,0


Pemeriksaan selesai. Data belum dihapus atau diubah.


### Interpretasi Hasil Pemeriksaan

Kombinasi delapan atribut pada Kandidat 1 menghasilkan 617.082 kombinasi unik dari total 1.694.907 baris. Hal ini menunjukkan bahwa kombinasi tersebut tidak mengidentifikasi setiap baris secara unik.

Kandidat 2 menggunakan 24 atribut dimensi dan menghasilkan 1.694.907 kombinasi unik, sama dengan jumlah seluruh baris. Namun, hasil ini hanya menunjukkan keunikan kombinasi atribut yang diperiksa dan belum membuktikan bahwa tidak terdapat duplikasi berdasarkan definisi bisnis dataset.

Karena kunci bisnis yang valid belum dapat dipastikan, penghapusan duplikasi ditunda untuk menghindari hilangnya catatan rantai pasok yang mungkin masih valid. Seluruh baris dipertahankan sementara sampai definisi satu catatan dapat dikonfirmasi melalui dokumentasi dataset.

In [30]:
# Data dipertahankan karena kunci bisnis belum terkonfirmasi.
# Tidak ada baris yang dihapus pada tahap ini.

total_baris_sebelum = df_outlier_handled.height

df_dedup = df_outlier_handled.clone()

total_baris_sesudah = df_dedup.height

print(f"Total baris sebelum: {total_baris_sebelum:,}")
print(f"Total baris sesudah: {total_baris_sesudah:,}")
print(f"Jumlah baris yang dihapus: {total_baris_sebelum - total_baris_sesudah:,}")

assert total_baris_sesudah == total_baris_sebelum

print("Validasi berhasil: seluruh baris dipertahankan.")
print("Deduplikasi aktual ditunda sampai kunci bisnis terkonfirmasi.")

Total baris sebelum: 1,694,907
Total baris sesudah: 1,694,907
Jumlah baris yang dihapus: 0
Validasi berhasil: seluruh baris dipertahankan.
Deduplikasi aktual ditunda sampai kunci bisnis terkonfirmasi.


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

Validasi data dasar dilakukan untuk memastikan bahwa data hasil tahap pembersihan memenuhi aturan yang telah ditentukan. Pemeriksaan mencakup jumlah baris, nilai kosong pada kolom numerik utama, serta nilai yang berada di luar batas hasil metode Interquartile Range (IQR) pada tahap penanganan outlier.

Selain itu, kolom volume diperiksa untuk mengidentifikasi nilai negatif karena volume rantai pasok secara umum tidak diharapkan bernilai negatif. Pemeriksaan ini penting karena batas bawah IQR yang diperoleh pada tahap sebelumnya dapat bernilai negatif.

Validasi dilakukan menggunakan DuckDB agar agregasi dan penghitungan pelanggaran dapat dilakukan secara efisien. Hasilnya digunakan untuk mengevaluasi kondisi data, bukan untuk mengubah atau menghapus baris secara otomatis.

In [31]:
# Mendaftarkan DataFrame hasil tahap deduplikasi ke DuckDB
connection.register('dataset_bersih', df_dedup)

# Memastikan variabel validasi dari tahap sebelumnya tersedia
assert KOLOM_NUMERIK in df_dedup.columns, (
    f"Kolom {KOLOM_NUMERIK} tidak ditemukan."
)

assert batas_bawah is not None and batas_atas is not None, (
    "Batas IQR belum tersedia. Jalankan tahap penanganan outlier terlebih dahulu."
)

# Validasi dasar berdasarkan kolom volume dan batas IQR
validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,

        SUM(
            CASE
                WHEN {KOLOM_NUMERIK} IS NULL THEN 1
                ELSE 0
            END
        ) AS pelanggaran_null,

        SUM(
            CASE
                WHEN {KOLOM_NUMERIK} < {batas_bawah}
                  OR {KOLOM_NUMERIK} > {batas_atas}
                THEN 1
                ELSE 0
            END
        ) AS pelanggaran_batas_iqr,

        SUM(
            CASE
                WHEN {KOLOM_NUMERIK} < 0 THEN 1
                ELSE 0
            END
        ) AS pelanggaran_volume_negatif

    FROM dataset_bersih
""").pl()

display(validasi)

total_baris,pelanggaran_null,pelanggaran_batas_iqr,pelanggaran_volume_negatif
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]"
1694907,0,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [32]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 1,694,907
Baris sesudah cleaning: 1,694,907
Data bersih tersimpan di: /home/jovyan/work/data/processed/dataset_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.